In [0]:
%sql
SELECT *  
FROM observatorio_dev.bronze.precio_bolsa
LIMIT 1;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS observatorio_dev.silver.precio_bolsa (
    id BIGINT,
    codigo_variable STRING,
    fecha_hora TIMESTAMP,
    codigo_duracion STRING,
    unidad_medida STRING,
    version STRING,
    valor DOUBLE,

    source_file_name STRING,
    source_file_path STRING,
    ingestion_timestamp TIMESTAMP,
    load_date DATE,

    silver_created_at TIMESTAMP,
    silver_updated_at TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality' = 'silver',
    'description' = 'Tabla Silver limpia y tipada del precio de bolsa de energía'
);

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    to_timestamp,
    current_timestamp,
    max as spark_max,
    lit,
    count,
    when
)

from delta.tables import DeltaTable

In [0]:
bronze_table = "observatorio_dev.bronze.precio_bolsa"
silver_table = "observatorio_dev.silver.precio_bolsa"

print("Bronze table:", bronze_table)
print("Silver table:", silver_table)

In [0]:
last_ingestion_timestamp = (
    spark.table(silver_table)
    .select(
        spark_max("ingestion_timestamp").alias("last_ingestion_timestamp")
    )
    .collect()[0]["last_ingestion_timestamp"]
)

print("Último ingestion_timestamp cargado en Silver:", last_ingestion_timestamp)

In [0]:
bronze_df = spark.table(bronze_table)

if last_ingestion_timestamp is not None:
    bronze_df = bronze_df.filter(
        col("ingestion_timestamp") > lit(last_ingestion_timestamp)
    )

print("Registros nuevos desde Bronze:", bronze_df.count())

display(bronze_df.limit(20))

In [0]:
silver_df = (
    bronze_df
    .select(
        col("id").cast("long").alias("id"),
        trim(upper(col("codigo_variable"))).alias("codigo_variable"),
        to_timestamp(col("fecha_hora")).alias("fecha_hora"),
        trim(upper(col("codigo_duracion"))).alias("codigo_duracion"),
        trim(upper(col("unidad_medida"))).alias("unidad_medida"),
        trim(upper(col("version"))).alias("version"),
        col("valor").cast("double").alias("valor"),

        col("source_file_name"),
        col("source_file_path"),
        col("ingestion_timestamp"),
        col("load_date"),

        current_timestamp().alias("silver_created_at"),
        current_timestamp().alias("silver_updated_at")
    )
    .dropDuplicates([
        "id",
        "fecha_hora",
        "codigo_variable",
        "codigo_duracion",
        "version"
    ])
)

print("Registros transformados para Silver:", silver_df.count())

display(silver_df.limit(20))

In [0]:
new_records_count = silver_df.count()

print("Registros nuevos a procesar:", new_records_count)

if new_records_count == 0:
    print("No hay registros nuevos para cargar en Silver.")

else:
    target = DeltaTable.forName(spark, silver_table)

    (
        target.alias("target")
        .merge(
            silver_df.alias("source"),
            """
            target.id = source.id
            AND target.fecha_hora = source.fecha_hora
            AND target.codigo_variable = source.codigo_variable
            AND target.codigo_duracion = source.codigo_duracion
            AND target.version = source.version
            """
        )
        .whenMatchedUpdate(set={
            "unidad_medida": "source.unidad_medida",
            "valor": "source.valor",
            "source_file_name": "source.source_file_name",
            "source_file_path": "source.source_file_path",
            "ingestion_timestamp": "source.ingestion_timestamp",
            "load_date": "source.load_date",
            "silver_updated_at": "source.silver_updated_at"
        })
        .whenNotMatchedInsert(values={
            "id": "source.id",
            "codigo_variable": "source.codigo_variable",
            "fecha_hora": "source.fecha_hora",
            "codigo_duracion": "source.codigo_duracion",
            "unidad_medida": "source.unidad_medida",
            "version": "source.version",
            "valor": "source.valor",
            "source_file_name": "source.source_file_name",
            "source_file_path": "source.source_file_path",
            "ingestion_timestamp": "source.ingestion_timestamp",
            "load_date": "source.load_date",
            "silver_created_at": "source.silver_created_at",
            "silver_updated_at": "source.silver_updated_at"
        })
        .execute()
    )

    print("MERGE ejecutado correctamente.")

In [0]:
%sql
SELECT *  
FROM observatorio_dev.silver.precio_bolsa
ORDER BY id

In [0]:
%sql
SELECT
    COUNT(*) AS registros,
    MIN(fecha_hora) AS fecha_minima,
    MAX(fecha_hora) AS fecha_maxima,
    COUNT(DISTINCT version) AS versiones_distintas
FROM observatorio_dev.silver.precio_bolsa;

In [0]:
%sql
SELECT
    COUNT(*) AS grupos_duplicados,
    COALESCE(SUM(repeticiones), 0) AS filas_en_grupos,
    COALESCE(SUM(repeticiones - 1), 0) AS filas_excedentes,
    COALESCE(MAX(repeticiones), 0) AS maximo_repeticiones
FROM (
    SELECT
        codigo_variable,
        fecha_hora,
        codigo_duracion,
        unidad_medida,
        version,
        valor,
        COUNT(*) AS repeticiones
    FROM observatorio_dev.silver.precio_bolsa
    GROUP BY
        codigo_variable,
        fecha_hora,
        codigo_duracion,
        unidad_medida,
        version,
        valor
    HAVING COUNT(*) > 1
);